In [ ]:
import torch
import numpy as np
import copy
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchdiffeq import odeint
from helpers import  objective, adj_lattice, criticalK,adjoint_init,   order_param,  adabelief_update, grad_estimate_vec, adabelief_update_manifold, combine_vector_field, AGM_ODE, ODE_Adjoint
import sys
sys.path.append('../../..')
from configs.v_agm_cfg import cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- 1. Modular Configuration ---

# --- 2. System Initialization ---
%store -r Astored
%store -r freqs
state_size = Astored.shape[1]

# Physical Constants
Kcrit_val, L_mat = criticalK(Astored, state_size, cfg.scale)
K_ctrl = 0.01 * Kcrit_val

# Tensor Factory
ctx = {'device': device, 'dtype': torch.float32}
A_torch = torch.as_tensor(Astored, **ctx)
L_torch = torch.as_tensor(L_mat, **ctx)
ts_exp = torch.linspace(0, cfg.t_total, cfg.t_size).to(device)
dt_val = ts_exp[1] - ts_exp[0]

# Initial State
ys_ran = torch.randn(cfg.batch_size, state_size, **ctx) / cfg.scale_kin
ys_ini = torch.zeros(cfg.batch_size, 1, **ctx)
ys_exp = torch.cat((ys_ran, ys_ini), dim=1)

# --- 3. Decomposed Control Setup ---
# Temporal part: (Time, Batch, 1)
temporal = torch.zeros(cfg.t_size, cfg.batch_size, 1, **ctx)

# Spatial part: (1, Batch, State_size) - normalized Rank-1 components
components = torch.ones(1, cfg.batch_size, state_size, **ctx)
components = components / torch.norm(components, dim=2, keepdim=True)

force_field = temporal * components

# Solvers
SML = AGM_ODE(cfg.batch_size, state_size, K_ctrl, A_torch, freqs, force_field, dt_val).to(device)
SMD = ODE_Adjoint(cfg.batch_size, state_size, K_ctrl, freqs, A_torch, force_field, ys_ran, dt_val, device).to(device)

# --- 4. Dual-Optimizer State ---
opt_spatial = {
    'm': torch.zeros_like(components), 'v': torch.zeros_like(components),
    'v_max': torch.zeros_like(components), 'step': 0
}
opt_temporal = {
    'm': torch.zeros_like(temporal), 'v': torch.zeros_like(temporal),
    'v_max': torch.zeros_like(temporal), 'step': 0
}

history = {"loss": [], "ste": [], "order_p": []}

# --- 5. Optimization Loop ---
pbar = tqdm(range(cfg.iterations), desc="Manifold Adjoint Opt")

for j in pbar:
    # A. Reconstruct Control & Forward Pass
    current_field = temporal * components
    SML.u = current_field
    
    with torch.no_grad():
        # Heun's method for better stability in oscillatory systems
        ys_tray = odeint(SML, ys_exp, ts_exp, method='heun2')

    # B. Loss Computation
    ys_phases = ys_tray[:, :, 0:state_size]
    obj = objective(ys_phases[-1], A_torch)
    kin = cfg.R_penalty * ys_tray[-1, :, -1]
    total_loss = obj + kin
    
    avg_loss = total_loss.mean().item()
    ord_p = torch.mean(order_param(ys_phases[-1], L_torch)).item()

    # C. Adjoint (Backward) Pass
    initials = adjoint_init(ys_phases[-1], state_size)
    SMD.x = ys_phases
    SMD.u = current_field
    
    with torch.no_grad():
        p_tray = odeint(SMD, initials, ts_exp, method='heun2')

    # D. Decomposed Gradient Estimation
    p_rev = torch.flip(p_tray, [0])
    # Extract component-wise and scalar-wise gradients
    comp_grad, scal_grad = grad_estimate_vec(p_rev, ys_phases, temporal, components, state_size, cfg.R_penalty, K_ctrl)
    
    # Aggregate gradients for temporal (scalar) and components (vector)
    # Temporal gets the sum across oscillators; components get the time-integrated effect
    beta_grad_scal = 2 * cfg.R_penalty * temporal + (K_ctrl * torch.sum(comp_grad, dim=2, keepdim=True))
    beta_grad_comp = Kcrit_val * torch.sum(scal_grad, dim=0, keepdim=True)

    # E. Manifold-Aware Optimizer Updates
    with torch.no_grad():
        # Update Temporal (Standard AdaBelief)
        temporal, opt_temporal['m'], opt_temporal['v'], opt_temporal['v_max'], opt_temporal['step'] = \
            adabelief_update(temporal, beta_grad_scal, opt_temporal['m'], opt_temporal['v'], 
                             opt_temporal['v_max'], opt_temporal['step'], lr=cfg.lr_base)

        # Update Components (Manifold AdaBelief - ensures normalization)
        components, opt_spatial['m'], opt_spatial['v'], opt_spatial['v_max'], opt_spatial['step'] = \
            adabelief_update_manifold(components, beta_grad_comp, opt_spatial['m'], opt_spatial['v'], 
                                      opt_spatial['v_max'], opt_spatial['step'], lr=cfg.lr_base)

    # F. Logging & Cleanup
    history["loss"].append(avg_loss)
    history["ste"].append(total_loss.std().item() / np.sqrt(cfg.batch_size))
    history["order_p"].append(ord_p)

    if j % 10 == 0:
        pbar.set_postfix({"Loss": f"{avg_loss:.4f}", "OrderP": f"{ord_p:.3f}"})
    
    if j % 500 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# --- 6. Final Diagnostics ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history["loss"], label="Total Cost")
plt.title("Convergence History")
plt.subplot(1, 2, 2)
plt.plot(history["order_p"], color='orange', label="Order Parameter")
plt.title("Synchronization Progress")
plt.show()